In [1]:
import os

IGNORAR = {"__pycache__", "data"}  # adicione aqui o que quiser ignorar

for root, dirs, files in os.walk(".."):
    dirs[:] = [d for d in dirs if not d.startswith(".") and d not in IGNORAR]
    level = root.replace(".", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        size_str = f"{size/1024/1024:.1f}MB" if size > 1024*1024 else f"{size/1024:.0f}KB"
        print(f"{indent}  {f}  ({size_str})")

../
  .gitignore  (0KB)
  README.md  (4KB)
  requirements.txt  (0KB)
  fase2/
    models/
      model_cha.pth  (77KB)
      model_turb.pth  (77KB)
    src/
      build_records.py  (4KB)
      dataset.py  (4KB)
      model.py  (2KB)
      train.py  (3KB)
      __init__.py  (0KB)
    test_output/
      result_chla.json  (0KB)
      result_turbidity.json  (0KB)
    track2_download_link_1/
      Guide to the Second Round_track2/
        Round2 Project Submission Manual_cn_0407.pdf  (2.5MB)
        Round2 Project Submission Manual_en_0407.pdf  (2.2MB)
        Space Intelligence for Clean Water Track_Task Description of Final Round_cn.pdf  (1.2MB)
        Space Intelligence for Clean Water Track_Task Description of Final Round_en.pdf  (1.1MB)
        test_input_sample/
          track2_cha_test_point.csv  (0KB)
          track2_turb_test_point.csv  (0KB)
          area8_images/
            area8_2024-01-15.tif  (60.4MB)
    track2_download_link_2/
      area6/
        track2_cha_train_point_

In [2]:
import rasterio
import pandas as pd
import numpy as np

# testando inspecionar tifs
for path in [
    "../fase2/track2_download_link_2/area6/area6_images/area6_2024-02-04.tif",
    "../fase2/track2_download_link_2/area6/area6_images/area6_2024-02-09.tif",
]:
    with rasterio.open(path) as src:
        print(f"\n{path.split('/')[-1]}")
        print(f"  bandas : {src.count}")
        print(f"  shape  : {src.shape}  (height x width)")
        print(f"  CRS    : {src.crs}")
        print(f"  res    : {src.res}")
        print(f"  dtype  : {src.dtypes[0]}")
        print(f"  bounds : {src.bounds}")
        # lê banda 1 e mostra range de valores
        b1 = src.read(1).astype(float)
        if src.nodata is not None:
            b1[b1 == src.nodata] = np.nan
        print(f"  min/max banda1: {np.nanmin(b1):.1f} / {np.nanmax(b1):.1f}")

# inspeciona csv de treino
for path in [
    "../fase2/track2_download_link_2/area6/track2_turb_train_point_area6.csv",
    "../fase2/track2_download_link_2/area6/track2_cha_train_point_area6.csv",
    "../fase2/track2_download_link_2/area7/track2_turb_train_point_area7.csv",
]:
    df = pd.read_csv(path)
    print(f"\n{path.split('/')[-1]}")
    print(df.head(3))
    print(f"  shape: {df.shape}")
    col = [c for c in df.columns if c not in ('filename','Lon','Lat')][0]
    print(f"  {col}: min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}")


area6_2024-02-04.tif
  bandas : 12
  shape  : (2500, 2130)  (height x width)
  CRS    : EPSG:4326
  res    : (0.00032559248826291, 0.00024343839999999944)
  dtype  : float32
  bounds : BoundingBox(left=-75.343552, bottom=41.280903, right=-74.65004, top=41.889499)
  min/max banda1: 0.0 / 1.2

area6_2024-02-09.tif
  bandas : 12
  shape  : (81, 76)  (height x width)
  CRS    : EPSG:4326
  res    : (9.009210526325786e-05, 6.664197530863667e-05)
  dtype  : float32
  bounds : BoundingBox(left=-75.217938, bottom=41.865009, right=-75.211091, top=41.870407)
  min/max banda1: 0.0 / 0.5

track2_turb_train_point_area6.csv
               filename        Lon        Lat  turb_value
0  area6_2024-01-15.tif -74.795580  41.309385         8.6
1  area6_2024-02-04.tif -74.795417  41.309264         3.1
2  area6_2024-02-14.tif -74.795108  41.309196         1.8
  shape: (31, 4)
  turb_value: min=0.50, max=8.60, mean=1.99

track2_cha_train_point_area6.csv
               filename        Lon        Lat  cha_val

In [3]:
import rasterio
import numpy as np
import pandas as pd

# 1. confirmar o que são as 12 bandas (nomes ou descrições se existirem)
with rasterio.open("../fase2/track2_download_link_2/area6/area6_images/area6_2024-02-04.tif") as src:
    print("Descrições das bandas:")
    for i, desc in enumerate(src.descriptions, 1):
        print(f"  banda {i}: {desc}")
    print("\nTags:", src.tags())

    # 2. pegar um ponto do CSV e ver se o pixel correspondente tem valores válidos
    df = pd.read_csv("../fase2/track2_download_link_2/area6/track2_turb_train_point_area6.csv")
    row = df[df['filename'] == 'area6_2024-02-04.tif'].iloc[0]
    lon, lat = row['Lon'], row['Lat']

    py, px = src.index(lon, lat)
    print(f"\nPonto: lon={lon}, lat={lat}")
    print(f"Pixel correspondente: row={py}, col={px}")
    print(f"Shape da imagem: {src.shape}")

    # extrai patch 11x11 centrado no ponto
    PATCH = 11
    half = PATCH // 2
    window = rasterio.windows.Window(px - half, py - half, PATCH, PATCH)
    patch = src.read(window=window)  # shape: (12, 11, 11)
    print(f"\nShape do patch: {patch.shape}")
    print(f"Valores banda 1 (pixel central): {patch[0, half, half]:.4f}")
    print(f"Valores todas as bandas no pixel central:")
    print(np.round(patch[:, half, half], 4))

Descrições das bandas:
  banda 1: None
  banda 2: None
  banda 3: None
  banda 4: None
  banda 5: None
  banda 6: None
  banda 7: None
  banda 8: None
  banda 9: None
  banda 10: None
  banda 11: None
  banda 12: None

Tags: {'AREA_OR_POINT': 'Area'}

Ponto: lon=-74.795417, lat=41.309264
Pixel correspondente: row=2383, col=1683
Shape da imagem: (2500, 2130)

Shape do patch: (12, 11, 11)
Valores banda 1 (pixel central): 0.0072
Valores todas as bandas no pixel central:
[0.0072 0.0109 0.0164 0.0157 0.0159 0.0146 0.0156 0.0181 0.0202 0.0823
 0.0452 0.0293]


In [4]:
import rasterio
import numpy as np
import pandas as pd
from pathlib import Path

# mapeamento de todos os pares (imagem, csv_turb, csv_cha)
AREAS = {
    'area1': {
        'img_dir': '../fase2/track2_download_link_5/area1/area1_images',
        'turb':    '../fase2/track2_download_link_5/area1/track2_turb_train_point_area1.csv',
        'cha':     '../fase2/track2_download_link_5/area1/track2_cha_train_point_area1.csv',
    },
    'area2': {
        'img_dir': '../fase2/track2_download_link_4/area2/area2_images',
        'turb':    '../fase2/track2_download_link_4/area2/track2_turb_train_point_area2.csv',
        'cha':     None,
    },
    'area3': {
        'img_dir': '../fase2/track2_download_link_3/area3/area3_images',
        'turb':    '../fase2/track2_download_link_3/area3/track2_turb_train_point_area3.csv',
        'cha':     None,
    },
    'area5': {
        'img_dir': '../fase2/track2_download_link_3/area5/area5_images',
        'turb':    '../fase2/track2_download_link_3/area5/track2_turb_train_point_area5.csv',
        'cha':     '../fase2/track2_download_link_3/area5/track2_cha_train_point_area5.csv',
    },
    'area6': {
        'img_dir': '../fase2/track2_download_link_2/area6/area6_images',
        'turb':    '../fase2/track2_download_link_2/area6/track2_turb_train_point_area6.csv',
        'cha':     '../fase2/track2_download_link_2/area6/track2_cha_train_point_area6.csv',
    },
    'area7': {
        'img_dir': '../fase2/track2_download_link_2/area7/area7_images',
        'turb':    '../fase2/track2_download_link_2/area7/track2_turb_train_point_area7.csv',
        'cha':     '../fase2/track2_download_link_2/area7/track2_cha_train_point_area7.csv',
    },
}

# 1. contagem total de amostras por alvo
print("=" * 50)
print("AMOSTRAS POR ÁREA E ALVO")
print("=" * 50)
total_turb, total_cha = 0, 0
for area, cfg in AREAS.items():
    n_turb, n_cha = 0, 0
    if cfg['turb']:
        n_turb = len(pd.read_csv(cfg['turb']))
        total_turb += n_turb
    if cfg['cha']:
        n_cha = len(pd.read_csv(cfg['cha']))
        total_cha += n_cha
    print(f"  {area}: turb={n_turb:>4}  cha={n_cha:>4}")
print(f"  TOTAL : turb={total_turb:>4}  cha={total_cha:>4}")

# 2. distribuição dos targets (com e sem log)
print("\n" + "=" * 50)
print("DISTRIBUIÇÃO DOS TARGETS")
print("=" * 50)
for target in ['turb', 'cha']:
    vals = []
    for area, cfg in AREAS.items():
        if cfg[target]:
            df = pd.read_csv(cfg[target])
            col = 'turb_value' if target == 'turb' else 'cha_value'
            vals.extend(df[col].dropna().tolist())
    vals = np.array(vals)
    print(f"\n  {target} (escala original):")
    print(f"    n={len(vals)}  min={vals.min():.2f}  max={vals.max():.2f}"
          f"  mean={vals.mean():.2f}  median={np.median(vals):.2f}  std={vals.std():.2f}")
    log_vals = np.log1p(vals)
    print(f"  {target} (log1p):")
    print(f"    min={log_vals.min():.3f}  max={log_vals.max():.3f}"
          f"  mean={log_vals.mean():.3f}  std={log_vals.std():.3f}")

# 3. verificar se todos os TIFs referenciados nos CSVs existem
print("\n" + "=" * 50)
print("ARQUIVOS FALTANDO")
print("=" * 50)
missing = 0
for area, cfg in AREAS.items():
    for target in ['turb', 'cha']:
        if not cfg[target]:
            continue
        df = pd.read_csv(cfg[target])
        for fname in df['filename'].unique():
            fpath = Path(cfg['img_dir']) / fname
            if not fpath.exists():
                print(f"  FALTA: {area}/{fname}")
                missing += 1
if missing == 0:
    print("  nenhum arquivo faltando")

# 4. verificar TIFs minúsculos (provavelmente inutilizáveis)
print("\n" + "=" * 50)
print("TIFs MUITO PEQUENOS (shape < 100x100)")
print("=" * 50)
for area, cfg in AREAS.items():
    for tif in sorted(Path(cfg['img_dir']).glob('*.tif')):
        with rasterio.open(tif) as src:
            h, w = src.shape
            if h < 100 or w < 100:
                print(f"  {area}/{tif.name}: {h}x{w}")

AMOSTRAS POR ÁREA E ALVO
  area1: turb= 211  cha=  91
  area2: turb= 200  cha=   0
  area3: turb= 247  cha=   0
  area5: turb=  98  cha=  28
  area6: turb=  31  cha=  12
  area7: turb= 204  cha= 174
  TOTAL : turb= 991  cha= 305

DISTRIBUIÇÃO DOS TARGETS

  turb (escala original):
    n=991  min=0.50  max=4000.00  mean=39.27  median=15.30  std=175.21
  turb (log1p):
    min=0.405  max=8.294  mean=2.808  std=1.138

  cha (escala original):
    n=305  min=0.01  max=75.90  mean=11.99  median=8.00  std=12.05
  cha (log1p):
    min=0.010  max=4.343  mean=2.134  std=0.986

ARQUIVOS FALTANDO
  nenhum arquivo faltando

TIFs MUITO PEQUENOS (shape < 100x100)
  area5/area5_2024-02-29.tif: 38x46
  area5/area5_2024-07-13.tif: 38x46
  area6/area6_2024-02-09.tif: 81x76
  area6/area6_2024-07-03.tif: 81x76
  area6/area6_2024-09-01.tif: 81x76


In [15]:
import os
import sys
from pathlib import Path

notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent 
src_path = str(project_root / "fase2" / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

import build_records
import dataset

In [6]:
import sys
sys.path.insert(0, '../fase2/src')

from build_records import build_records
from dataset import WaterQualityDataset
import torch

# monta records
records_turb = build_records('turb')
records_cha  = build_records('cha')

# cria datasets
ds_turb = WaterQualityDataset(records_turb, target='turb', augment=False)
ds_cha  = WaterQualityDataset(records_cha,  target='cha',  augment=False)

# testa um item
feat, label = ds_turb[0]
print(f"feature shape : {feat.shape}")       # esperado: torch.Size([28])
print(f"feature range : {feat.min():.3f} – {feat.max():.3f}")
print(f"label (log1p) : {label.item():.3f}") # esperado: valor entre 0.4 e 8.3

# verifica distribuição dos labels no dataset completo
import numpy as np
labels_turb = np.array([ds_turb[i][1].item() for i in range(len(ds_turb))])
labels_cha  = np.array([ds_cha[i][1].item()  for i in range(len(ds_cha))])
print(f"\nturb log1p — mean={labels_turb.mean():.3f}  std={labels_turb.std():.3f}"
      f"  min={labels_turb.min():.3f}  max={labels_turb.max():.3f}")
print(f"cha  log1p — mean={labels_cha.mean():.3f}  std={labels_cha.std():.3f}"
      f"  min={labels_cha.min():.3f}  max={labels_cha.max():.3f}")

[turb] total válidos : 963
[turb] skip tif peq : 4
[turb] skip oob     : 24
[turb] skip null    : 0
[cha] total válidos : 283
[cha] skip tif peq : 0
[cha] skip oob     : 22
[cha] skip null    : 0
feature shape : torch.Size([33])
feature range : -2.505 – 0.866
label (log1p) : 2.821

turb log1p — mean=2.827  std=1.129  min=0.405  max=8.294
cha  log1p — mean=2.178  std=0.962  min=0.010  max=4.343


In [27]:
import sys, pathlib

# remove todos os .pyc do src para forçar recompilação
src = pathlib.Path('../fase2/src')
for pyc in src.rglob('*.pyc'):
    pyc.unlink()
    print(f'removido: {pyc}')

pycache = src / '__pycache__'
if pycache.exists():
    import shutil
    shutil.rmtree(pycache)
    print('removido: __pycache__')

# limpa módulos do sys.modules
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['train', 'model', 'dataset', 'build_records']):
        del sys.modules[mod]

print('pronto')

removido: ..\fase2\src\__pycache__\build_records.cpython-312.pyc
removido: ..\fase2\src\__pycache__\dataset.cpython-312.pyc
removido: ..\fase2\src\__pycache__\train.cpython-312.pyc
removido: __pycache__
pronto


In [12]:
# no 03_exploracao_fase2.ipynb, célula de treino
import sys
sys.path.insert(0, '../fase2/src')

from build_records import build_records
from train import train_target
from pathlib import Path

Path('../fase2/models').mkdir(exist_ok=True)

records_turb = build_records('turb')
records_cha  = build_records('cha')

r2_turb = train_target(records_turb, 'turb', '../fase2/models/model_turb.pth')
r2_cha  = train_target(records_cha,  'cha',  '../fase2/models/model_cha.pth')

[turb] total válidos : 963
[turb] skip tif peq : 4
[turb] skip oob     : 24
[turb] skip null    : 0
[cha] total válidos : 283
[cha] skip tif peq : 0
[cha] skip oob     : 22
[cha] skip null    : 0
[turb] treino=769  val=194
  epoch  10 | loss=0.3678 | val RMSE(log)=2.0562  R²=-5.8958 | RMSE(orig)=31.53
    -> checkpoint salvo (R²=-5.8958)
  epoch  20 | loss=0.2713 | val RMSE(log)=1.7369  R²=-3.9204 | RMSE(orig)=31.11
    -> checkpoint salvo (R²=-3.9204)
  epoch  30 | loss=0.2287 | val RMSE(log)=1.5526  R²=-2.9318 | RMSE(orig)=67.73
    -> checkpoint salvo (R²=-2.9318)
  epoch  40 | loss=0.2380 | val RMSE(log)=1.4081  R²=-2.2339 | RMSE(orig)=43.62
    -> checkpoint salvo (R²=-2.2339)
  epoch  50 | loss=0.2066 | val RMSE(log)=1.4717  R²=-2.5326 | RMSE(orig)=59.45
  epoch  60 | loss=0.2006 | val RMSE(log)=1.4510  R²=-2.4338 | RMSE(orig)=57.20
  epoch  70 | loss=0.1762 | val RMSE(log)=1.4560  R²=-2.4578 | RMSE(orig)=67.02
  epoch  80 | loss=0.1813 | val RMSE(log)=1.3722  R²=-2.0709 | RMSE(o

In [ ]:
# no notebook — simula o ambiente do container localmente
import sys, os
sys.path.insert(0, '../fase2/src')

# aponta para os dados de teste locais
os.environ['TEST_INPUT']  = '../fase2/track2_download_link_1/Guide to the Second Round_track2/test_input_sample'
os.environ['TEST_OUTPUT'] = '../fase2/test_output'

import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from dataset import load_patch, compute_indices, HALF, N_BANDS
from model import WaterQualityMLP

input_dir  = Path(os.environ['TEST_INPUT'])
output_dir = Path(os.environ['TEST_OUTPUT'])
output_dir.mkdir(exist_ok=True)
img_dir    = input_dir / 'area8_images'

def load_model(path):
    m = WaterQualityMLP()
    m.load_state_dict(torch.load(path, map_location='cpu'))
    m.eval()
    return m

model_turb = load_model('../fase2/models/model_turb.pth')
model_cha  = load_model('../fase2/models/model_cha.pth')

for target, csv_name, out_name in [
    ('turb', 'track2_turb_test_point.csv', 'result_turbidity.json'),
    ('cha',  'track2_cha_test_point.csv',  'result_chla.json'),
]:
    df     = pd.read_csv(input_dir / csv_name)
    result = {}
    for _, row in df.iterrows():
        key   = f"{row['filename']}_{row['Lon']}_{row['Lat']}"
        patch = load_patch(img_dir / row['filename'], row['Lon'], row['Lat'])
        if patch is None:
            result[key] = [15.3 if target == 'turb' else 8.0]
            continue
        import re
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', row['filename'])
        month = int(match.group(2)) if match else 6
        month_sin = np.float32(np.sin(2 * np.pi * month / 12))
        month_cos = np.float32(np.cos(2 * np.pi * month / 12))
        lat_norm  = np.float32((row['Lat'] - 42.5) / 5.0)
        lon_norm  = np.float32((row['Lon'] + 82)   / 10.0)

        feats = np.concatenate([patch[:, HALF, HALF],
                                compute_indices(patch),
                                patch.reshape(N_BANDS, -1).std(-1),
                                [month_sin, month_cos, lat_norm, lon_norm]])
        
        x     = torch.tensor(feats).unsqueeze(0)
        with torch.no_grad():
            val = float(np.expm1(model_turb(x, target='turb').item()
                                 if target == 'turb' else
                                 model_cha(x, target='cha').item()))
        result[key] = [round(val, 4)]

    with open(output_dir / out_name, 'w') as f:
        json.dump(result, f, indent=2)
    print(f"{out_name}: {len(result)} pontos")
    print(f"  exemplo: {next(iter(result.items()))}")
    print(f"  range: {min(v[0] for v in result.values()):.2f}"
          f" – {max(v[0] for v in result.values()):.2f}")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x29 and 33x128)

In [16]:
import sys, pathlib, shutil

src = pathlib.Path('../fase2/src')
for pyc in src.rglob('*.pyc'):
    pyc.unlink()
if (src / '__pycache__').exists():
    shutil.rmtree(src / '__pycache__')
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['train', 'model', 'dataset', 'build_records']):
        del sys.modules[mod]

from build_records import build_records
from train import train_target
from pathlib import Path

Path('../fase2/models').mkdir(exist_ok=True)
records_turb = build_records('turb')
records_cha  = build_records('cha')
r2_turb = train_target(records_turb, 'turb', '../fase2/models/model_turb.pth')
r2_cha  = train_target(records_cha,  'cha',  '../fase2/models/model_cha.pth')

[turb] total válidos : 963
[turb] skip tif peq : 4
[turb] skip oob     : 24
[turb] skip null    : 0
[cha] total válidos : 283
[cha] skip tif peq : 0
[cha] skip oob     : 22
[cha] skip null    : 0
[turb] treino=769  val=194
  epoch  10 | loss=0.3088 | val RMSE(log)=1.8022  R²=-4.2975 | RMSE(orig)=31.27
    -> checkpoint salvo (R²=-4.2975)
  epoch  20 | loss=0.2775 | val RMSE(log)=1.6532  R²=-3.4576 | RMSE(orig)=31.84
    -> checkpoint salvo (R²=-3.4576)
  epoch  30 | loss=0.2491 | val RMSE(log)=1.5938  R²=-3.1432 | RMSE(orig)=34.19
    -> checkpoint salvo (R²=-3.1432)
  epoch  40 | loss=0.2321 | val RMSE(log)=1.4660  R²=-2.5054 | RMSE(orig)=30.70
    -> checkpoint salvo (R²=-2.5054)
  epoch  50 | loss=0.2104 | val RMSE(log)=1.3401  R²=-1.9290 | RMSE(orig)=33.40
    -> checkpoint salvo (R²=-1.9290)
  epoch  60 | loss=0.1878 | val RMSE(log)=1.3638  R²=-2.0335 | RMSE(orig)=38.79
  epoch  70 | loss=0.1880 | val RMSE(log)=1.3865  R²=-2.1352 | RMSE(orig)=31.22
  epoch  80 | loss=0.1919 | val 

In [17]:
import sys, pathlib, shutil
sys.path.insert(0, '../fase2/src')

from build_records import build_records
from dataset import compute_indices, HALF, N_BANDS
import numpy as np

records = build_records('turb')

# monta matrix de features e targets manualmente
X, y = [], []
for r in records:
    patch = r['patch']
    center  = patch[:, HALF, HALF]
    indices = compute_indices(patch)
    spatial = patch.reshape(N_BANDS, -1).std(-1)
    month     = r.get('month', 6)
    month_sin = np.float32(np.sin(2 * np.pi * month / 12))
    month_cos = np.float32(np.cos(2 * np.pi * month / 12))
    lat_norm  = np.float32((r.get('lat', 42) - 42.5) / 5.0)
    lon_norm  = np.float32((r.get('lon', -82) + 82)  / 10.0)
    feats = np.concatenate([center, indices, spatial,
                            [month_sin, month_cos, lat_norm, lon_norm]])
    X.append(feats)
    y.append(np.log1p(r['label']))

X = np.array(X)
y = np.array(y)

# correlação de Pearson de cada feature com o target
nomes = ([f'b{i}' for i in range(12)] +
         ['ndwi','ndti','ndci','mndwi','nir_red'] +
         [f'std_b{i}' for i in range(12)] +
         ['month_sin','month_cos','lat','lon'])

corrs = [np.corrcoef(X[:, i], y)[0, 1] for i in range(X.shape[1])]
ranked = sorted(zip(corrs, nomes), key=lambda x: abs(x[0]), reverse=True)

print("Top 10 features por correlação com log(turb):")
for corr, nome in ranked[:10]:
    print(f"  {nome:>12}: {corr:+.3f}")

print(f"\nMáxima correlação absoluta: {max(abs(c) for c,_ in ranked):.3f}")

# teste rápido com modelo linear
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score
scores = cross_val_score(Ridge(), X, y, cv=5, scoring='r2')
print(f"\nRidge CV R² (5-fold): {scores.mean():.3f} ± {scores.std():.3f}")

[turb] total válidos : 963
[turb] skip tif peq : 4
[turb] skip oob     : 24
[turb] skip null    : 0
Top 10 features por correlação com log(turb):
           lat: -0.365
           lon: -0.329
       nir_red: +0.291
          ndwi: -0.260
     month_cos: -0.156
         mndwi: +0.155
        std_b7: -0.152
            b3: +0.139
        std_b6: -0.132
            b4: +0.132

Máxima correlação absoluta: 0.365

Ridge CV R² (5-fold): -0.273 ± 0.586


In [23]:
import sys, pathlib, shutil

src = pathlib.Path('../fase2/src')
for pyc in src.rglob('*.pyc'):
    pyc.unlink()
if (src / '__pycache__').exists():
    shutil.rmtree(src / '__pycache__')
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['train', 'model', 'dataset', 'build_records', 'infer']):
        del sys.modules[mod]

sys.path.insert(0, '../fase2/src')
from build_records import build_records
from train import train_target
from pathlib import Path

records_turb = build_records('turb')
records_cha  = build_records('cha')
r2_turb = train_target(records_turb, 'turb', '../fase2/models/model_turb.joblib')
r2_cha  = train_target(records_cha,  'cha',  '../fase2/models/model_cha.joblib')

[turb] total válidos : 963
[turb] skip tif peq : 4
[turb] skip oob     : 24
[turb] skip null    : 0
[cha] total válidos : 283
[cha] skip tif peq : 0
[cha] skip oob     : 22
[cha] skip null    : 0

=== turb ===
  p99 cap=406.5 — 953 pontos restantes
  features: (953, 49)
  CV R² (5-fold): -0.388 ± 0.464
  salvo: ../fase2/models/model_turb.joblib
  top 10 importâncias:
          ratio4: 0.128
              b3: 0.094
       month_cos: 0.055
            ndwi: 0.054
             b11: 0.039
          std_b9: 0.034
              b4: 0.031
              b2: 0.031
          std_b7: 0.025
            ndci: 0.024

=== cha ===
  p99 cap=51.4 — 280 pontos restantes
  features: (280, 49)
  CV R² (5-fold): -0.593 ± 0.826
  salvo: ../fase2/models/model_cha.joblib
  top 10 importâncias:
          std_b9: 0.060
         mean_b8: 0.056
              b7: 0.051
          ratio0: 0.048
         std_b10: 0.042
              b9: 0.039
         std_b11: 0.037
          ratio5: 0.036
          ratio4: 0.035
   

In [24]:
# no notebook — testa se área8 está dentro do range geográfico do treino
import sys
sys.path.insert(0, '../fase2/src')
from build_records import build_records
import numpy as np

records = build_records('turb')
lats = np.array([r['lat'] for r in records])
lons = np.array([r['lon'] for r in records])
print(f"treino lat: {lats.min():.2f} – {lats.max():.2f}")
print(f"treino lon: {lons.min():.2f} – {lons.max():.2f}")

# coordenadas de área8 (do CSV de teste que já vimos)
print(f"\nárea8 lat: ~44.5 – 45.5")
print(f"área8 lon: ~-122.8 – -122.2")

[turb] total válidos : 963
[turb] skip tif peq : 4
[turb] skip oob     : 24
[turb] skip null    : 0
treino lat: 29.78 – 42.61
treino lon: -109.03 – -74.78

área8 lat: ~44.5 – 45.5
área8 lon: ~-122.8 – -122.2


In [25]:
import sys, pathlib, shutil
sys.path.insert(0, '../fase2/src')

from build_records import build_records
from train import build_features
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

records = build_records('turb')

# remove p99
vals = np.array([r['label'] for r in records])
cap  = np.quantile(vals, 0.99)
records = [r for r in records if r['label'] <= cap]

X, y = build_features(records)
areas = np.array([r['area'] for r in records])

print("Leave-one-area-out R²:")
for area in sorted(set(areas)):
    train_mask = areas != area
    val_mask   = areas == area
    if val_mask.sum() < 5:
        continue
    gbr = GradientBoostingRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, min_samples_leaf=5, random_state=42)
    gbr.fit(X[train_mask], y[train_mask])
    pred = gbr.predict(X[val_mask])
    r2   = r2_score(y[val_mask], pred)
    rmse_orig = np.sqrt(np.mean(
        (np.expm1(pred) - np.expm1(y[val_mask]))**2))
    print(f"  {area}: R²={r2:+.3f}  RMSE_orig={rmse_orig:.1f}"
          f"  n={val_mask.sum()}")

[turb] total válidos : 963
[turb] skip tif peq : 4
[turb] skip oob     : 24
[turb] skip null    : 0
Leave-one-area-out R²:
  area1: R²=-1.443  RMSE_orig=31.0  n=210
  area2: R²=+0.169  RMSE_orig=63.5  n=192
  area3: R²=-0.004  RMSE_orig=26.4  n=246
  area5: R²=-1.529  RMSE_orig=21.2  n=84
  area6: R²=-6.756  RMSE_orig=7.8  n=27
  area7: R²=-0.835  RMSE_orig=28.8  n=194


In [28]:
import sys
sys.path.insert(0, '../fase2/src')
from build_records import build_records
import numpy as np
from collections import defaultdict

records = build_records('turb')
records_cha = build_records('cha')

# mediana por mês — captura sazonalidade sem depender de coordenadas
for target, recs in [('turb', records), ('cha', records_cha)]:
    por_mes = defaultdict(list)
    for r in recs:
        por_mes[r['month']].append(r['label'])
    
    print(f"\n{target} — mediana por mês:")
    medianas = {}
    for mes in sorted(por_mes):
        med = np.median(por_mes[mes])
        medianas[mes] = med
        print(f"  mês {mes:>2}: n={len(por_mes[mes]):>3}  mediana={med:.2f}")
    
    # global como fallback se mês não tiver dados
    print(f"  global:  mediana={np.median([r['label'] for r in recs]):.2f}")

[turb] total válidos : 963
[turb] skip tif peq : 4
[turb] skip oob     : 24
[turb] skip null    : 0
[cha] total válidos : 283
[cha] skip tif peq : 0
[cha] skip oob     : 22
[cha] skip null    : 0

turb — mediana por mês:
  mês  1: n= 44  mediana=7.10
  mês  2: n= 75  mediana=6.20
  mês  3: n= 69  mediana=12.60
  mês  4: n=107  mediana=31.60
  mês  5: n= 96  mediana=32.60
  mês  6: n=137  mediana=17.40
  mês  7: n=150  mediana=10.30
  mês  8: n=144  mediana=16.45
  mês  9: n=141  mediana=9.00
  global:  mediana=15.40

cha — mediana por mês:
  mês  1: n= 10  mediana=4.94
  mês  2: n= 25  mediana=6.50
  mês  3: n= 28  mediana=13.30
  mês  4: n= 33  mediana=14.70
  mês  5: n= 21  mediana=5.30
  mês  6: n= 34  mediana=6.35
  mês  7: n= 48  mediana=7.56
  mês  8: n= 40  mediana=11.14
  mês  9: n= 44  mediana=10.49
  global:  mediana=8.30


In [29]:
import sys, pathlib, shutil

src = pathlib.Path('../fase2/src')
for pyc in src.rglob('*.pyc'):
    pyc.unlink()
if (src / '__pycache__').exists():
    shutil.rmtree(src / '__pycache__')
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ['train', 'model', 'dataset', 'build_records', 'infer']):
        del sys.modules[mod]

import json
import numpy as np
import pandas as pd
import joblib
import rasterio
import re
from pathlib import Path
from collections import defaultdict
sys.path.insert(0, '../fase2/src')
from dataset import compute_indices, HALF, N_BANDS, PATCH_SIZE
from infer import extract_features, predict_csv, get_fallback

input_dir  = Path('../fase2/track2_download_link_1/Guide to the Second Round_track2/test_input_sample')
output_dir = Path('../fase2/test_output')
output_dir.mkdir(exist_ok=True)
img_dir    = input_dir / 'area8_images'

model_turb = joblib.load('../fase2/models/model_turb.joblib')
model_cha  = joblib.load('../fase2/models/model_cha.joblib')

for target, csv_name, out_name in [
    ('turb', 'track2_turb_test_point.csv', 'result_turbidity.json'),
    ('cha',  'track2_cha_test_point.csv',  'result_chla.json'),
]:
    result = predict_csv(input_dir / csv_name, img_dir, 
                         model_turb if target == 'turb' else model_cha, 
                         target)
    with open(output_dir / out_name, 'w') as f:
        json.dump(result, f, indent=2)
    print(f"{out_name}: {len(result)} pontos")
    print(f"  exemplo : {next(iter(result.items()))}")
    print(f"  range   : {min(v[0] for v in result.values()):.2f}"
          f" – {max(v[0] for v in result.values()):.2f}")

result_turbidity.json: 3 pontos
  exemplo : ('area8_2024-01-15.tif_-122.823396_44.498493', [11.0223])
  range   : 4.25 – 20.28
result_chla.json: 2 pontos
  exemplo : ('area8_2024-01-15.tif_-122.669408_45.517321', [6.7173])
  range   : 6.27 – 6.72
